# M4 · Connect independently running programs

A public interface translates requests into ordinary operations. The supplied TestClient is an in-process HTTP caller: it runs real routes without requiring a separate server; M8 teaches its testing role.

**Start:** Before class, from `teaching-snippets/`, run `uv sync --extra dev` and select its Python environment as the notebook kernel. Notebook-editor/kernel setup is separate classroom preparation. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Caller used here:** FastAPI’s TestClient uses HTTPX in process. It exercises decoding, routing, and response encoding, but does not establish live-network transport or timeout behavior. External requests are never sent automatically. For the separate-process requests demonstration, use the optional activity linked at the end.

## Exploration 1 — build and observe the mechanism

### 1. Exchange values through JSON

A Python dictionary and JSON text are different representations. Encoding and decoding preserve agreed values, not shared object identity.

**Before running:** Predict the result or state change.

In [ ]:
import json
payload = {"items": [{"product_id": "1", "quantity": 2}]}
message = json.dumps(payload)
received = json.loads(message)
print(type(payload).__name__, type(message).__name__)
print(received == payload, received is payload)


**Change and explain:** Change the local payload after encoding; inspect received again.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The values initially match, but the decoded dictionary is a different object.

</details>

### 2. Translate a named record

Records express meaning inside Python. Translate them to ordinary structures before exchanging them.

**Before running:** Predict the result or state change.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class Product:
    product_id: str
    name: str
    price_paise: int

cable = Product("1", "USB-C Cable", 49900)
print(json.dumps(asdict(cable)))


**Change and explain:** Add the bottle as another Product; encode it the same way.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The JSON object contains the product fields, not a live Python instance.

</details>

### 3. Choose a representation JSON supports

An external format has a smaller vocabulary than Python. Unsupported values need a deliberate translation.

**Before running:** Predict the result or state change.

In [ ]:
from datetime import date
try:
    json.dumps({"ordered_on": date(2026, 10, 12)})
except TypeError as error:
    print(type(error).__name__)
print(json.dumps({"ordered_on": date(2026, 10, 12).isoformat()}))


**Change and explain:** Explain which side must interpret the agreed date string.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A date object fails directly; an explicit ISO date string can be encoded.

</details>

### 4. Publish a catalog operation

A route exposes a callable through an HTTP interface. The supplied in-process caller avoids server setup during exploration.

**Before running:** Predict the result or state change.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
products = {"1": asdict(cable), "3": {"product_id": "3", "name": "Water Bottle", "price_paise": 89900}}
app = FastAPI()

@app.get("/products")
def list_products(q: str = ""):
    return [p for p in products.values() if q.casefold() in p["name"].casefold()]

client = TestClient(app)
response = client.get("/products")
print(response.status_code, response.json())


**Change and explain:** Trace the route, result, status, and decoded response.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Status 200 and the two products. TestClient runs real routes but does not test a live network.

</details>

### 5. Use a query to narrow selection

A query parameter selects data without changing the catalog.

**Before running:** Predict the result or state change.

In [ ]:
response = client.get("/products", params={"q": "CABLE"})
print(response.request.url)
print(response.json())
print(len(products))


**Change and explain:** Try a query with no match.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

One cable is returned; the catalog still has two products.

</details>

### 6. Map a missing resource to a response

A path identifies the requested resource. The boundary translates absence into a deliberate protocol outcome.

**Before running:** Predict the result or state change.

In [ ]:
@app.get("/products/{product_id}")
def get_product(product_id: str):
    if product_id not in products:
        raise HTTPException(status_code=404, detail="Unknown product")
    return products[product_id]

print(client.get("/products/1").status_code)
missing = client.get("/products/999")
print(missing.status_code, missing.json())


**Change and explain:** Explain why a 404 differs from receiving no HTTP response.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The known product returns 200; the missing one returns 404 and a detail body.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Describe the order body

A typed body gives the handler a known shape. FastAPI already uses Pydantic internally for these dataclasses.

**Before running:** Predict the result or state change.

In [ ]:
@dataclass
class RequestedItem:
    product_id: str
    quantity: int

@dataclass
class OrderRequest:
    items: list[RequestedItem]

request = OrderRequest([RequestedItem("1", 2)])
print(asdict(request))


**Change and explain:** Keep the outer items object when adding another requested line.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The request shape remains {"items": [...]}; M5 adds explicit strict constraints.

</details>

### 8. Keep shopping rules behind the boundary

An ordinary operation computes authoritative prices and returns a result. It does not need to know HTTP.

**Before running:** Predict the result or state change.

In [ ]:
from uuid import uuid4

def create_order(items):
    if not items:
        raise ValueError("Cannot checkout an empty cart")
    lines = []
    for item in items:
        product = products[item.product_id]
        if item.quantity <= 0:
            raise ValueError("Quantity must be positive")
        lines.append({"product_id": item.product_id, "name": product["name"],
                      "unit_price_paise": product["price_paise"], "quantity": item.quantity})
    total = sum(line["unit_price_paise"] * line["quantity"] for line in lines)
    return {"order_id": uuid4().hex, "lines": lines, "total_paise": total}

print(create_order(request.items)["total_paise"])


**Change and explain:** Explain why the request does not supply a trusted total.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two cables total 99800; an unknown product fails at catalog lookup.

</details>

### 9. Delegate creation and retain the result

A thin handler translates domain failures and delegates the calculation. Here orders remain in process memory.

**Before running:** Predict the result or state change.

In [ ]:
orders = {}

@app.post("/orders", status_code=201)
def place_order(request: OrderRequest):
    try:
        order = create_order(request.items)
    except KeyError:
        raise HTTPException(404, "Unknown product")
    except ValueError as error:
        raise HTTPException(422, str(error))
    orders[order["order_id"]] = order
    return order

created = client.post("/orders", json=payload)
print(created.status_code, created.json()["total_paise"])


**Change and explain:** Change quantity to 3 and submit again; each call creates an independent order.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Status 201, total 99800, and a generated order ID.

</details>

### 10. Retrieve through the public identity

The returned identity connects later requests to the created result. Local variable names are not a public protocol.

**Before running:** Predict the result or state change.

In [ ]:
@app.get("/orders/{order_id}")
def get_order(order_id: str):
    if order_id not in orders:
        raise HTTPException(404, "Unknown order")
    return orders[order_id]

order = created.json()
retrieved = client.get(f"/orders/{order['order_id']}")
print(retrieved.status_code, retrieved.json() == order)


**Change and explain:** Request an invented order ID and inspect its outcome.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Status 200 and matching values; rebuilding this app resets its memory store.

</details>

### 11. Interpret status before success data

A response can describe rejection. A caller should check the status before treating the body as a product.

**Before running:** Predict the result or state change.

In [ ]:
import httpx

def fetch_product(caller, product_id):
    response = caller.get(f"/products/{product_id}")
    response.raise_for_status()
    return response.json()

print(fetch_product(client, "1")["name"])
try:
    fetch_product(client, "999")
except httpx.HTTPStatusError as error:
    print("Rejected:", error.response.status_code)


**Change and explain:** Explain why this supplied caller uses httpx error types; the live requests caller uses requests exceptions.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The known name is displayed; the missing ID raises the caller’s HTTP error.

</details>

### 12. Inspect an outgoing live request without sending it

The requests library can prepare JSON for a separate process. Sending it also needs a running server and an explicit timeout.

**Before running:** Predict the result or state change.

In [ ]:
import requests
prepared = requests.Request("POST", "http://127.0.0.1:8000/orders", json=payload).prepare()
print(prepared.method, prepared.url)
print(prepared.headers["Content-Type"])
print(json.loads(prepared.body))
client.close()


**Change and explain:** In the optional live demonstration, locate timeout, status checking, and response decoding in the supplied caller.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The JSON request is prepared; no external network call is made by this notebook.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M4 lab guide](../../projects/nano-zon/labs/m04/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Optional live-server activity

The [teaching-snippet guide](../../teaching-snippets/README.md#api-sequence) supplies the matching uvicorn server and separate Python caller. This activity requires two terminals and deliberate launch; it is not a prerequisite for the cells above. External order submissions can create additional orders. A timeout does not establish whether the server created an order, so do not blindly repeat a POST.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.